# 7주차 실습: Tiled MatMul과 Triton 컴파일 파이프라인

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/tensor2silicon/blob/main/notebooks/week7_triton_matmul.ipynb)

[학습 자료](https://github.com/Jake-Song/tensor2silicon/blob/main/week7/README.md)

GPU: Colab·Kaggle에서 GPU 런타임(예: Tesla T4, L4, A100)을 선택하고 순서대로 실행한다.

CPU: 아래 `INTERPRET=True`로 설정하면 GPU 없이도 2차원 포인터 오프셋, 마스킹, K축 누적, Epilogue 활성화 결합의 정확도를 확인할 수 있다(단, CPU 해석기는 NumPy로 순차 실행하므로 오토튜닝 벤치마크와 `.asm` 컴파일 산출물 확인은 건너뛴다).

첫 손계산 셀은 Python 표준 라이브러리만으로 실행할 수 있다.

In [ ]:
# 1. 9x9 출력 블록 그리드에서 Row-major vs Grouped Ordering (GROUP_SIZE_M=3) 입력 타일 수 비교
num_pid_m, num_pid_n, k_blocks = 9, 9, 9
GROUP_SIZE_M = 3
num_pid_in_group = GROUP_SIZE_M * num_pid_n

rm_coords = []
gr_coords = []
for pid in range(9):
    rm_coords.append((pid // num_pid_n, pid % num_pid_n))
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m
    gr_coords.append((pid_m, pid_n))

rm_tiles = (len({m for m, _ in rm_coords}) + len({n for _, n in rm_coords})) * k_blocks
gr_tiles = (len({m for m, _ in gr_coords}) + len({n for _, n in gr_coords})) * k_blocks
print("pid 0~8 Row-major 좌표:", rm_coords, "-> 필요 입력 타일 수:", rm_tiles)
print("pid 0~8 Grouped   좌표:", gr_coords, "-> 필요 입력 타일 수:", gr_tiles)

# 2. M=N=K=4096 FP16 입력 / FP32 누산 행렬곱의 3개 후보 Config 손계산
M = N = K = 4096
F = 2 * M * N * K
P, BW = 80e12, 1e12  # 80 TFLOPs/s, 1 TB/s (I_ridge = 80 FLOPs/Byte)
Q_out = 2 * M * N
t_comp_ms = (F / P) * 1e3

candidates = [
    ("후보 1 (64x32x32, s=5, w=2)", 64, 32, 32, 5, 2),
    ("후보 2 (128x128x32, s=4, w=4)", 128, 128, 32, 4, 4),
    ("후보 3 (128x256x64, s=3, w=8)", 128, 256, 64, 3, 8),
]

print(f"\n공통 연산량 F = {F:,} FLOPs, t_comp = {t_comp_ms:.6f} ms, 출력 쓰기 = {Q_out:,} Byte")
for name, BM, BN, BK, stages, warps in candidates:
    grid = (M // BM) * (N // BN)
    k_iters = K // BK
    Q_in = 2 * M * N * K * (1 / BN + 1 / BM)
    Q_total = Q_in + Q_out
    I = F / Q_total
    t_mem_ms = (Q_total / BW) * 1e3
    t_ideal_ms = max(t_comp_ms, t_mem_ms)
    bound = "Memory-bound" if I < (P / BW) else "Compute-bound"
    smem_1stage = 2 * (BM * BK + BK * BN)
    smem_pipe_kib = (smem_1stage * stages) / 1024
    acc_kib = (4 * BM * BN) / 1024
    limit_64k = "OutOfResources (>64KiB)" if smem_pipe_kib > 64 else "OK (<=64KiB)"
    print(
        f"{name}: grid=({grid},), K루프={k_iters}, Q={Q_total/1e9:.3f} GB, "
        f"I={I:.2f} FLOPs/B, T_ideal={t_ideal_ms:.3f} ms ({bound}), "
        f"SMEM={smem_pipe_kib:.1f} KiB [{limit_64k}], Acc={acc_kib:.1f} KiB"
    )

## 1. 환경 설정

`TRITON_INTERPRET`와 `TRITON_PRINT_AUTOTUNING`은 `triton`을 import하기 전에 설정한다.

In [ ]:
import os

INTERPRET = False  # CPU 해석 모드: True로 변경 시 런타임 재시작 후 처음부터 실행
os.environ["TRITON_INTERPRET"] = "1" if INTERPRET else "0"
os.environ["TRITON_PRINT_AUTOTUNING"] = "1"

import json
import platform
import sys
import torch
import triton
import triton.language as tl
import triton.testing

if not INTERPRET and not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임을 선택하거나 재시작 후 INTERPRET=True로 실행하세요.")

device = torch.device("cpu" if INTERPRET else "cuda")
torch.manual_seed(0)

cc = None
smem_limit_bytes = None
if not INTERPRET:
    props = torch.cuda.get_device_properties(device)
    cc = f"{props.major}.{props.minor}"
    smem_limit_bytes = getattr(props, "shared_memory_per_block_optin", props.shared_memory_per_block)

env_info = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "triton": triton.__version__,
    "cuda_build": torch.version.cuda,
    "gpu": None if INTERPRET else torch.cuda.get_device_name(device),
    "compute_capability": cc,
    "shared_memory_per_block_optin_kib": None if smem_limit_bytes is None else smem_limit_bytes / 1024,
    "mode": "CPU interpreter" if INTERPRET else "GPU",
}
print(json.dumps(env_info, ensure_ascii=False, indent=2))

## 2. 2차원 Tiled MatMul 커널 (단일 Config 및 Autotune 버전)

- `offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak`로 `[BM, BK]` 2차원 포인터 블록을 만든다.
- `GROUP_SIZE_M`으로 출력 블록 실행 순서를 묶어 L2 캐시 재사용을 높인다.
- K축 루프에서 `tl.dot(a, b, accumulator)`로 FP32 누산기에 누적하고, 루프 직후 선택적으로 `leaky_relu`를 온칩에서 결합(Epilogue Fusion)한 뒤 FP16으로 변환해 저장한다.

[공식 Tutorial 03: Matrix Multiplication](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html)

In [ ]:
@triton.jit
def leaky_relu(x):
    return tl.where(x >= 0, x, 0.01 * x)


@triton.jit
def matmul_kernel_single(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
    ACTIVATION: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        k_remaining = K - k * BLOCK_SIZE_K
        a = tl.load(a_ptrs, mask=offs_k[None, :] < k_remaining, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < k_remaining, other=0.0)
        accumulator = tl.dot(a, b, accumulator)
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    if ACTIVATION == "leaky_relu":
        accumulator = leaky_relu(accumulator)
    c = accumulator.to(tl.float16)

    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)


def get_cuda_autotune_config():
    return [
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 64, "GROUP_SIZE_M": 8}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8}, num_stages=5, num_warps=2),
        triton.Config({"BLOCK_SIZE_M": 32, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8}, num_stages=5, num_warps=2),
    ]


matmul_kernel_autotuned = triton.autotune(
    configs=get_cuda_autotune_config(),
    key=["M", "N", "K"],
)(matmul_kernel_single)


def matmul_single(
    a, b, activation="",
    block_m=64, block_n=32, block_k=32, group_m=8,
    num_stages=5, num_warps=2,
    return_kernel=False,
):
    assert a.ndim == 2 and b.ndim == 2 and a.shape[1] == b.shape[0]
    assert a.dtype == b.dtype == torch.float16
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = (triton.cdiv(M, block_m) * triton.cdiv(N, block_n),)
    compiled = matmul_kernel_single[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        BLOCK_SIZE_M=block_m,
        BLOCK_SIZE_N=block_n,
        BLOCK_SIZE_K=block_k,
        GROUP_SIZE_M=group_m,
        ACTIVATION=activation,
        num_stages=num_stages,
        num_warps=num_warps,
    )
    if return_kernel:
        return c, compiled
    return c


def matmul_autotuned(a, b, activation=""):
    assert a.ndim == 2 and b.ndim == 2 and a.shape[1] == b.shape[0]
    assert a.dtype == b.dtype == torch.float16
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = lambda META: (triton.cdiv(M, META["BLOCK_SIZE_M"]) * triton.cdiv(N, META["BLOCK_SIZE_N"]),)
    matmul_kernel_autotuned[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        ACTIVATION=activation,
    )
    return c

## 3. 정확도 확인 (2의 거듭제곱 & 비거듭제곱 경계 마스킹 & Epilogue Fusion)

- `512 × 512 × 512` FP16 입력과 `500 × 600 × 300`(2의 거듭제곱이 아닌 크기)에서 `torch.matmul(a, b)`와 `torch.allclose(..., atol=1e-2, rtol=0)`를 통과하는지 확인한다.
- CPU 해석 모드에서는 실행 시간을 고려해 작은 크기(`64 × 64 × 64` 및 `50 × 60 × 30`)로 동일한 경계 마스킹과 `leaky_relu` 결합을 검증한다.

In [ ]:
ATOL, RTOL = 1e-2, 0.0
correctness = []

test_shapes = [(64, 64, 64), (50, 60, 30)] if INTERPRET else [(512, 512, 512), (500, 600, 300)]
fn_to_test = matmul_single if INTERPRET else matmul_autotuned

for M_t, N_t, K_t in test_shapes:
    a = torch.randn((M_t, K_t), device=device, dtype=torch.float16)
    b = torch.randn((K_t, N_t), device=device, dtype=torch.float16)

    # 1) 기본 행렬곱 검증
    triton_out = fn_to_test(a, b)
    torch_out = torch.matmul(a, b)
    ok = torch.allclose(triton_out, torch_out, atol=ATOL, rtol=RTOL)
    max_diff = (triton_out - torch_out).abs().max().item()
    assert ok, f"Shape {(M_t, N_t, K_t)} mismatch! max_diff={max_diff}"
    correctness.append({"shape": [M_t, N_t, K_t], "activation": "none", "allclose": ok, "max_diff": max_diff})

    # 2) Epilogue Fusion (leaky_relu) 검증
    triton_act = fn_to_test(a, b, activation="leaky_relu")
    torch_act = torch.nn.functional.leaky_relu(torch.matmul(a, b), negative_slope=0.01)
    ok_act = torch.allclose(triton_act, torch_act, atol=ATOL, rtol=RTOL)
    max_diff_act = (triton_act - torch_act).abs().max().item()
    assert ok_act, f"Shape {(M_t, N_t, K_t)} leaky_relu mismatch! max_diff={max_diff_act}"
    correctness.append({"shape": [M_t, N_t, K_t], "activation": "leaky_relu", "allclose": ok_act, "max_diff": max_diff_act})

print("정확도 검증 통과:")
for item in correctness:
    print(" ", item)

## 4. 컴파일 산출물(`.metadata` · `ttir` · `ttgir` · `ptx`) 열어보기

단일 Config(`BLOCK_SIZE_M=64, BLOCK_SIZE_N=32, BLOCK_SIZE_K=32, num_stages=5, num_warps=2`)로 컴파일된 커널 객체에서 다음을 직접 확인한다.

1. `.metadata` (`shared`, `num_warps`, `num_stages`) 및 레지스터 수(`.n_regs`)
2. `.asm['ttir']`: 스레드·메모리 배치가 정해지지 않은 블록 수준 `tt.load`, `scf.for`, `tt.dot`, `tt.store`
3. `.asm['ttgir']`: `#ttg.blocked`, `#ttg.nvmma_shared` / `#ttg.shared`, `#ttg.dot_op` 레이아웃과 `local_alloc` 파이프라이닝
4. `.asm['ptx']`: NVIDIA 가상 어셈블리의 `mma.sync`(또는 아키텍처별 행렬곱 명령)와 `cp.async` 명령

In [ ]:
ir_summary = {}
if INTERPRET:
    print("CPU 해석 모드: 컴파일 파이프라인을 우회하므로 .metadata 및 .asm 확인을 생략합니다.")
else:
    a_ir = torch.randn((512, 512), device=device, dtype=torch.float16)
    b_ir = torch.randn((512, 512), device=device, dtype=torch.float16)
    _, compiled_kernel = matmul_single(
        a_ir, b_ir,
        block_m=64, block_n=32, block_k=32, group_m=8,
        num_stages=5, num_warps=2,
        return_kernel=True,
    )

    meta = compiled_kernel.metadata
    ir_summary = {
        "shared_bytes": getattr(meta, "shared", None),
        "num_warps": getattr(meta, "num_warps", None),
        "num_stages": getattr(meta, "num_stages", None),
        "n_regs": getattr(compiled_kernel, "n_regs", None),
        "n_spills": getattr(compiled_kernel, "n_spills", None),
        "asm_keys": list(compiled_kernel.asm.keys()),
    }
    print("=== CompiledKernel Metadata ===")
    print(json.dumps(ir_summary, ensure_ascii=False, indent=2))

    def grep_lines(text, keywords, max_lines=12):
        matched = [line.strip() for line in text.splitlines() if any(k in line for k in keywords)]
        return matched[:max_lines]

    for stage, kw in [
        ("ttir", ["tt.get_program_id", "tt.load", "scf.for", "tt.dot", "tt.store"]),
        ("ttgir", ["#ttg.", "local_alloc", "async_copy", "tt.dot", "convert_layout"]),
        ("ptx", [".target", "cp.async", "ldmatrix", "mma.sync"]),
    ]:
        if stage in compiled_kernel.asm:
            print(f"\n=== .asm['{stage}'] 핵심 줄 발췌 ===")
            for ln in grep_lines(compiled_kernel.asm[stage], kw):
                print(" ", ln)

## 5. 오토튜닝 및 성능(TFLOPs/s) 벤치마크

- `M = N = K = 4096` FP16 입력에서 단일 Config(후보 1: `64×32×32`, 후보 2: `128×128×32`), 오토튜닝된 Triton 커널, 그리고 `torch.matmul`(cuBLAS)의 실행 시간(ms)과 **TFLOPs/s**를 비교한다.
- $\text{TFLOPs/s} = \dfrac{2 \times M \times N \times K \times 10^{-12}}{t_{\text{ms}} \times 10^{-3}}$

In [ ]:
benchmark_results = []
best_config_str = None

if INTERPRET:
    print("CPU 해석 모드: GPU 성능 벤치마크 생략")
else:
    M = N = K = 4096
    a_bench = torch.randn((M, K), device=device, dtype=torch.float16)
    b_bench = torch.randn((K, N), device=device, dtype=torch.float16)
    reference = torch.matmul(a_bench, b_bench)
    flops = 2.0 * M * N * K

    providers = {
        "triton_cand1 (64x32x32, s=5, w=2)": lambda: matmul_single(
            a_bench, b_bench, block_m=64, block_n=32, block_k=32, group_m=8, num_stages=5, num_warps=2
        ),
        "triton_cand2 (128x128x32, s=4, w=4)": lambda: matmul_single(
            a_bench, b_bench, block_m=128, block_n=128, block_k=32, group_m=8, num_stages=4, num_warps=4
        ),
        "triton_autotuned": lambda: matmul_autotuned(a_bench, b_bench),
        "torch.matmul (cuBLAS)": lambda: torch.matmul(a_bench, b_bench),
    }

    for name, fn in providers.items():
        try:
            for _ in range(3):
                out = fn()
            assert torch.allclose(out, reference, atol=ATOL, rtol=RTOL), name
            torch.cuda.synchronize()
            ms = triton.testing.do_bench(fn, warmup=25, rep=100, return_mode="median")
            tflops = flops * 1e-12 / (ms * 1e-3)
            benchmark_results.append({"provider": name, "ms_median": ms, "tflops_per_s": tflops})
            print(f"{name:38s} {ms:8.4f} ms  {tflops:7.2f} TFLOPs/s")
        except Exception as e:
            print(f"{name:38s} 실행 불가 ({type(e).__name__}: {e})")
            benchmark_results.append({"provider": name, "error": str(e)})

    if hasattr(matmul_kernel_autotuned, "best_config"):
        best_config_str = str(matmul_kernel_autotuned.best_config)
        print("\n선택된 Best Autotune Config:", best_config_str)

    record = {
        "environment": env_info,
        "shape": [M, N, K],
        "input_dtype": "float16",
        "accumulator_dtype": "float32",
        "best_autotune_config": best_config_str,
        "ir_metadata_cand1": ir_summary,
        "correctness": correctness,
        "results": benchmark_results,
    }
    from pathlib import Path
    Path("week7_results.json").write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print("환경·IR 메타데이터·측정 결과 저장: week7_results.json")

## 확인하고 함께 이야기하기

1. `offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak`가 어떻게 `[BM, BK]` 2차원 포인터 블록이 되는지, 그리고 K 루프마다 포인터가 어떻게 이동하는지 설명할 수 있는가?
2. 9×9 출력 블록에서 `GROUP_SIZE_M = 3`이 처음 9개 블록의 입력 타일 수를 왜 90개에서 54개로 줄이는지 설명할 수 있는가?
3. 내 GPU의 블록당 최대 Shared Memory 용량은 몇 KiB이며, 후보 3(`128×256×64, num_stages=3`, SMEM 144 KiB)이 실행 가능한가?
4. `.asm['ttir']`와 `.asm['ttgir']`에서 `tl.dot`과 `tl.load`가 어떻게 다르게 표현되는가?